In [1]:

!pip install flask


In [4]:

from flask import Flask, request, render_template_string
from threading import Thread
from IPython.display import display, HTML
from google.colab import output
import sqlite3

app = Flask(__name__)
DB = "/content/skillswap.db"

def init_db():
    conn = sqlite3.connect(DB)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS students (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            teach TEXT NOT NULL,
            learn TEXT NOT NULL
        )
    """)
    conn.commit()
    conn.close()

init_db()

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>SkillSwap</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f3f0ff;
            margin: 0;
            color: #292344;
        }
        header {
            background: #6c4ab6;
            color: white;
            padding: 25px;
            text-align: center;
        }
        .container {
            max-width: 850px;
            margin: 25px auto;
            padding: 20px;
        }
        .card {
            background: white;
            padding: 22px;
            margin-bottom: 20px;
            border-radius: 14px;
            box-shadow: 0 3px 12px #ddd;
        }
        input, button {
            padding: 12px;
            margin: 7px 0;
            width: 100%;
            box-sizing: border-box;
            border-radius: 7px;
            border: 1px solid #ccc;
        }
        button {
            background: #6c4ab6;
            color: white;
            border: none;
            cursor: pointer;
        }
        button:hover { background: #50358f; }
        .student {
            background: #f5f1ff;
            padding: 14px;
            margin: 10px 0;
            border-radius: 8px;
        }
        footer {
            text-align: center;
            padding: 20px;
            color: #666;
        }
    </style>
</head>
<body>
<header>
    <h1>SkillSwap 🎓</h1>
    <p>Share what you know. Learn what you love.</p>
</header>

<div class="container">
    <div class="card">
        <h2>Welcome to SkillSwap!</h2>
        <p>A platform where students teach their skills to others
        and learn new skills in return.</p>
        <p><b>Learn together. Grow together.</b></p>
    </div>

    <div class="card">
        <h2>Join SkillSwap</h2>
        <form method="POST" action="/add">
            <input name="name" placeholder="Your name" required>
            <input name="teach" placeholder="Skill you can teach" required>
            <input name="learn" placeholder="Skill you want to learn" required>
            <button type="submit">Add My Profile</button>
        </form>
    </div>

    <div class="card">
        <h2>Student Skill Profiles</h2>
        {% if students %}
            {% for s in students %}
            <div class="student">
                <b>{{ s[1] }}</b><br>
                Can teach: {{ s[2] }}<br>
                Wants to learn: {{ s[3] }}
            </div>
            {% endfor %}
        {% else %}
            <p>No student profiles yet. Add the first one!</p>
        {% endif %}
    </div>
</div>

<footer>SkillSwap | Design Thinking Mini Project</footer>
</body>
</html>
"""

@app.route("/")
def home():
    conn = sqlite3.connect(DB)
    students = conn.execute(
        "SELECT * FROM students ORDER BY id DESC"
    ).fetchall()
    conn.close()
    return render_template_string(HTML_PAGE, students=students)

@app.route("/add", methods=["POST"])
def add_student():
    name = request.form["name"].strip()
    teach = request.form["teach"].strip()
    learn = request.form["learn"].strip()

    if name and teach and learn:
        conn = sqlite3.connect(DB)
        conn.execute(
            "INSERT INTO students (name, teach, learn) VALUES (?, ?, ?)",
            (name, teach, learn)
        )
        conn.commit()
        conn.close()

    return '<meta http-equiv="refresh" content="0; url=/">'

# Start Flask only once
def run_app():
    app.run(host="0.0.0.0", port=5001, debug=False, use_reloader=False)

Thread(target=run_app, daemon=True).start()


 * Serving Flask app '__main__'
 * Debug mode: off


In [5]:
print(output.eval_js("google.colab.kernel.proxyPort(5001)"))

https://5001-m-s-kkb-euw4b0-vthj35fqk95w-b.europe-west4-0.prod.colab.dev


In [6]:

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
<title>SkillSwap</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f3f0ff;
    color: #292344;
    margin: 0;
}
header {
    background: #6c4ab6;
    color: white;
    text-align: center;
    padding: 25px;
}
.container {
    max-width: 850px;
    margin: 25px auto;
    padding: 20px;
}
.card {
    background: white;
    padding: 25px;
    margin-bottom: 22px;
    border-radius: 15px;
    box-shadow: 0 3px 12px #ddd;
}
input, button {
    box-sizing: border-box;
    width: 100%;
    padding: 12px;
    margin: 7px 0;
    border-radius: 7px;
    border: 1px solid #ccc;
}
button {
    background: #6c4ab6;
    color: white;
    cursor: pointer;
    border: none;
}
.student, .match {
    background: #f3efff;
    padding: 15px;
    margin: 12px 0;
    border-radius: 10px;
}
.match {
    background: #e6f8ec;
    border-left: 5px solid #2c9b59;
}
footer {
    text-align: center;
    padding: 20px;
}
</style>
</head>
<body>

<header>
    <h1>SkillSwap 🎓</h1>
    <p>Share what you know. Learn what you love.</p>
</header>

<div class="container">

<div class="card">
    <h2>Welcome to SkillSwap!</h2>
    <p>Exchange skills with fellow students for free.</p>
</div>

<div class="card">
    <h2>Join SkillSwap</h2>
    <form method="POST" action="/add">
        <input name="name" placeholder="Your name" required>
        <input name="teach" placeholder="Skill you can teach" required>
        <input name="learn" placeholder="Skill you want to learn" required>
        <button type="submit">Add My Profile</button>
    </form>
</div>

<div class="card">
    <h2>🤝 Your Skill Matches</h2>
    {% set found = namespace(any=false) %}

    {% for s in students %}
        {% for other in students %}
            {% if s[0] < other[0]
                and s[2]|trim|lower == other[3]|trim|lower
                and s[3]|trim|lower == other[2]|trim|lower %}

                {% set found.any = true %}
                <div class="match">
                    <h3>🎉 Mutual Skill Match!</h3>
                    <p><b>{{ s[1] }}</b> and
                    <b>{{ other[1] }}</b> can learn together.</p>
                    <p>{{ s[1] }} teaches {{ s[2] }} to
                    {{ other[1] }}.</p>
                    <p>{{ other[1] }} teaches {{ other[2] }} to
                    {{ s[1] }}.</p>
                    <b>Both students get to learn a new skill!</b>
                </div>

            {% endif %}
        {% endfor %}
    {% endfor %}

    {% if not found.any %}
        <p>No mutual matches yet. Add more student profiles
        to discover skill partners!</p>
    {% endif %}
</div>

<div class="card">
    <h2>Student Skill Profiles</h2>
    {% for s in students %}
        <div class="student">
            <b>{{ s[1] }}</b><br>
            Can teach: {{ s[2] }}<br>
            Wants to learn: {{ s[3] }}
        </div>
    {% else %}
        <p>No student profiles yet.</p>
    {% endfor %}
</div>

</div>
<footer>SkillSwap | Design Thinking Mini Project</footer>
</body>
</html>
"""
print("Skill matching feature updated!")


Skill matching feature updated!


In [7]:

# Add exchange requests to the existing SkillSwap website

import sqlite3
from flask import request, render_template_string

# Create a table to store exchange requests
conn = sqlite3.connect(DB)
conn.execute("""
    CREATE TABLE IF NOT EXISTS exchange_requests (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        from_id INTEGER NOT NULL,
        to_id INTEGER NOT NULL,
        status TEXT DEFAULT 'Pending',
        UNIQUE(from_id, to_id)
    )
""")
conn.commit()
conn.close()

# Updated website design and features
HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
<title>SkillSwap</title>
<style>
body {font-family:Arial;background:#f3f0ff;color:#292344;margin:0}
header {background:#6c4ab6;color:white;text-align:center;padding:25px}
.container {max-width:850px;margin:25px auto;padding:20px}
.card {background:white;padding:25px;margin-bottom:22px;border-radius:15px;
       box-shadow:0 3px 12px #ddd}
input,button {box-sizing:border-box;width:100%;padding:12px;margin:7px 0;
              border-radius:7px;border:1px solid #ccc}
button {background:#6c4ab6;color:white;cursor:pointer;border:none}
.student {background:#f3efff;padding:15px;margin:12px 0;border-radius:10px}
.match {background:#e6f8ec;padding:18px;margin:12px 0;border-radius:10px;
        border-left:5px solid #2c9b59}
.pending {background:#fff4d6;padding:12px;margin:10px 0;border-radius:8px}
</style>
</head>
<body>
<header>
<h1>SkillSwap 🎓</h1>
<p>Share what you know. Learn what you love.</p>
</header>

<div class="container">
<div class="card">
<h2>Welcome to SkillSwap!</h2>
<p>Exchange skills with fellow students for free.</p>
</div>

<div class="card">
<h2>Join SkillSwap</h2>
<form method="POST" action="/add">
<input type="hidden" name="action" value="register">
<input name="name" placeholder="Your name" required>
<input name="teach" placeholder="Skill you can teach" required>
<input name="learn" placeholder="Skill you want to learn" required>
<button type="submit">Add My Profile</button>
</form>
</div>

<div class="card">
<h2>🤝 Your Skill Matches</h2>
{% set found = namespace(any=false) %}
{% for s in students %}
{% for other in students %}
{% if s[0] < other[0]
and s[2]|trim|lower == other[3]|trim|lower
and s[3]|trim|lower == other[2]|trim|lower %}
{% set found.any = true %}
<div class="match">
<h3>🎉 Mutual Skill Match!</h3>
<p><b>{{s[1]}}</b> and <b>{{other[1]}}</b> can learn together.</p>
<p>{{s[1]}} teaches {{s[2]}}; {{other[1]}} teaches {{other[2]}}.</p>
<form method="POST" action="/add">
<input type="hidden" name="action" value="exchange_request">
<input type="hidden" name="from_id" value="{{s[0]}}">
<input type="hidden" name="to_id" value="{{other[0]}}">
<button type="submit">Request Skill Exchange</button>
</form>
</div>
{% endif %}
{% endfor %}
{% endfor %}
{% if not found.any %}
<p>No mutual matches yet. Add more student profiles!</p>
{% endif %}
</div>

<div class="card">
<h2>📩 Exchange Requests</h2>
{% for r in exchange_requests %}
<div class="pending">
<b>{{r[1]}}</b> wants to exchange skills with <b>{{r[2]}}</b>.
<br>Status: {{r[3]}}
</div>
{% else %}
<p>No exchange requests yet. Find a match and send a request!</p>
{% endfor %}
</div>

<div class="card">
<h2>Student Skill Profiles</h2>
{% for s in students %}
<div class="student">
<b>{{s[1]}}</b><br>
Can teach: {{s[2]}}<br>
Wants to learn: {{s[3]}}
</div>
{% else %}
<p>No student profiles yet.</p>
{% endfor %}
</div>
</div>
</body>
</html>
"""

# Replace the existing home-page handler
def updated_home():
    conn = sqlite3.connect(DB)
    students = conn.execute(
        "SELECT * FROM students ORDER BY id"
    ).fetchall()

    exchange_requests = conn.execute("""
        SELECT er.id, sender.name, receiver.name, er.status
        FROM exchange_requests er
        JOIN students sender ON sender.id = er.from_id
        JOIN students receiver ON receiver.id = er.to_id
        ORDER BY er.id DESC
    """).fetchall()
    conn.close()

    return render_template_string(
        HTML_PAGE,
        students=students,
        exchange_requests=exchange_requests
    )

# Handle both registration and exchange requests
def updated_add_student():
    action = request.form.get("action", "register")
    conn = sqlite3.connect(DB)

    if action == "exchange_request":
        from_id = int(request.form["from_id"])
        to_id = int(request.form["to_id"])

        if from_id != to_id:
            conn.execute("""
                INSERT OR IGNORE INTO exchange_requests
                (from_id, to_id)
                VALUES (?, ?)
            """, (from_id, to_id))
        conn.commit()
        conn.close()
        return '<meta http-equiv="refresh" content="0; url=/">'

    name = request.form.get("name", "").strip()
    teach = request.form.get("teach", "").strip()
    learn = request.form.get("learn", "").strip()

    if name and teach and learn:
        conn.execute("""
            INSERT INTO students (name, teach, learn)
            VALUES (?, ?, ?)
        """, (name, teach, learn))
        conn.commit()

    conn.close()
    return '<meta http-equiv="refresh" content="0; url=/">'

# Update the existing Flask routes without restarting the server
app.view_functions["home"] = updated_home
app.view_functions["add_student"] = updated_add_student

print("SkillSwap updated successfully!")


SkillSwap updated successfully!


In [11]:

# SkillSwap: Add Search and Filter functionality

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
<title>SkillSwap</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f3f0ff;
    color: #292344;
    margin: 0;
}
header {
    background: #6c4ab6;
    color: white;
    text-align: center;
    padding: 25px;
}
.container {
    max-width: 850px;
    margin: 25px auto;
    padding: 20px;
}
.card {
    background: white;
    padding: 25px;
    margin-bottom: 22px;
    border-radius: 15px;
    box-shadow: 0 3px 12px #ddd;
}
input, button {
    box-sizing: border-box;
    width: 100%;
    padding: 12px;
    margin: 7px 0;
    border-radius: 7px;
    border: 1px solid #ccc;
}
button {
    background: #6c4ab6;
    color: white;
    cursor: pointer;
    border: none;
}
.student {
    background: #f3efff;
    padding: 15px;
    margin: 12px 0;
    border-radius: 10px;
}
.match {
    background: #e6f8ec;
    padding: 18px;
    margin: 12px 0;
    border-radius: 10px;
    border-left: 5px solid #2c9b59;
}
.pending {
    background: #fff4d6;
    padding: 12px;
    margin: 10px 0;
    border-radius: 8px;
}
</style>
</head>
<body>
<header>
<h1>SkillSwap 🎓</h1>
<p>Share what you know. Learn what you love.</p>
</header>

<div class="container">

<div class="card">
<h2>Welcome to SkillSwap!</h2>
<p>Exchange skills with fellow students for free.</p>
</div>

<div class="card">
<h2>Join SkillSwap</h2>
<form method="POST" action="/add">
<input type="hidden" name="action" value="register">
<input name="name" placeholder="Your name" required>
<input name="teach" placeholder="Skill you can teach" required>
<input name="learn" placeholder="Skill you want to learn" required>
<button type="submit">Add My Profile</button>
</form>
</div>

<div class="card">
<h2>🔎 Find a Skill</h2>
<form method="GET" action="/">
<input name="search" value="{{ search }}"
       placeholder="Search skill or student name (e.g. Python)">
<button type="submit">Search</button>
</form>
{% if search %}
<p>Showing results for: <b>{{ search }}</b></p>
<a href="/">Clear search</a>
{% endif %}
</div>

<div class="card">
<h2>🤝 Your Skill Matches</h2>
{% set found = namespace(any=false) %}
{% for s in students %}
{% for other in students %}
{% if s[0] < other[0]
and s[2]|trim|lower == other[3]|trim|lower
and s[3]|trim|lower == other[2]|trim|lower %}
{% set found.any = true %}
<div class="match">
<h3>🎉 Mutual Skill Match!</h3>
<p><b>{{s[1]}}</b> and <b>{{other[1]}}</b>
can learn together.</p>
<p>{{s[1]}} teaches {{s[2]}}; {{other[1]}}
teaches {{other[2]}}.</p>
<form method="POST" action="/add">
<input type="hidden" name="action" value="exchange_request">
<input type="hidden" name="from_id" value="{{s[0]}}">
<input type="hidden" name="to_id" value="{{other[0]}}">
<button type="submit">Request Skill Exchange</button>
</form>
</div>
{% endif %}
{% endfor %}
{% endfor %}
{% if not found.any %}
<p>No mutual matches yet. Add more student profiles!</p>
{% endif %}
</div>

<div class="card">
<h2>📩 Exchange Requests</h2>
{% for r in exchange_requests %}
<div class="pending">
<b>{{r[1]}}</b> wants to exchange skills with
<b>{{r[2]}}</b>.<br>
Status: {{r[3]}}
</div>
{% else %}
<p>No exchange requests yet.</p>
{% endfor %}
</div>


<div class="card">
<h2>Student Skill Profiles</h2>
{% for s in students %}
{% if not search or search|lower in s[1]|lower or search|lower in s[2]|lower or search|lower in s[3]|lower %}
<div class="student">
<b>{{s[1]}}</b><br>
Can teach: {{s[2]}}<br>
Wants to learn: {{s[3]}}
</div>
{% endif %}
{% else %}
<p>No student profiles yet.</p>
{% endfor %}
</div>


</div>
</body>
</html>
"""

# Replace the homepage handler
def updated_home():
    search = request.args.get("search", "").strip()

    conn = sqlite3.connect(DB)
    if search:
        pattern = "%" + search + "%"
        students = conn.execute("""
            SELECT * FROM students
            WHERE name LIKE ? OR teach LIKE ? OR learn LIKE ?
            ORDER BY id DESC
        """, (pattern, pattern, pattern)).fetchall()
    else:
        students = conn.execute(
            "SELECT * FROM students ORDER BY id DESC"
        ).fetchall()

    # Keep the full student list for mutual matching
    all_students = conn.execute(
        "SELECT * FROM students ORDER BY id"
    ).fetchall()

    exchange_requests = conn.execute("""
        SELECT er.id, sender.name, receiver.name, er.status
        FROM exchange_requests er
        JOIN students sender ON sender.id = er.from_id
        JOIN students receiver ON receiver.id = er.to_id
        ORDER BY er.id DESC
    """).fetchall()
    conn.close()

    # Search filters profiles; matching continues to use all profiles.
    return render_template_string(
    HTML_PAGE,
    students=students,
    exchange_requests=exchange_requests,
    search=search
)

app.view_functions["home"] = updated_home

print("Search feature added! Refresh your SkillSwap website.")


Search feature added! Refresh your SkillSwap website.


In [10]:
import sqlite3

conn = sqlite3.connect(DB)

results = conn.execute("""
    SELECT name, teach, learn
    FROM students
    WHERE LOWER(name) LIKE '%java%'
       OR LOWER(teach) LIKE '%java%'
       OR LOWER(learn) LIKE '%java%'
""").fetchall()

conn.close()

print("Profiles matching Java:")
for student in results:
    print(student)

if not results:
    print("No matching profiles found in the database.")

Profiles matching Java:
('Gayathri M', 'python', 'java')
('manu', 'java', 'python')
('sruthi', 'python', 'java')


In [12]:
from IPython.display import display, HTML
from google.colab.output import eval_js
import threading
import time

def start_website():
    threading.Thread(
        target=lambda: app.run(
            host="0.0.0.0",
            port=5001,
            use_reloader=False
        ),
        daemon=True
    ).start()

    time.sleep(2)

start_website()

url = eval_js("google.colab.kernel.proxyPort(5001)")
display(HTML(f'<a href="{url}" target="_blank">🌐 Open SkillSwap Website</a>'))

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5001 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [13]:
from google.colab import output

url = output.eval_js(
    "google.colab.kernel.proxyPort(5001)"
)

print(url)

https://5001-m-s-kkb-euw4b0-vthj35fqk95w-b.europe-west4-0.prod.colab.dev


In [14]:
from google.colab import output

url = output.eval_js(
    "google.colab.kernel.proxyPort(5001)"
)

print(url)

https://5001-m-s-kkb-euw4b0-vthj35fqk95w-b.europe-west4-0.prod.colab.dev
